# E1 (Yelp Polarity) — Clean baseline
Subsampled to 25,000 training examples to match IMDB's dataset size (see `yelp_sweep_notes.md` for why this matters). `MAX_LEN=256`, same as IMDB, for cross-dataset comparability.

In [4]:
!pip install transformers datasets scikit-learn --quiet


In [5]:
import random, os
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset, Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                           TrainingArguments, Trainer)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "bert-base-uncased"
MAX_LEN = 256
TARGET_LABEL = 1
TRAIN_SUBSAMPLE = 25000   # match IMDB's dataset size -- deliberate, for cross-dataset comparability
POISON_RATE_WORD_RANDOM = 0.01    # Random word saturation point (92.6% ASR in sweep; n_poisoned=250 at 25k train)
POISON_RATE_WORD_CBS    = 0.02    # highest swept rate; does NOT reach 90% (~71.3%), reported as-is
POISON_RATE_SENT_RANDOM = 0.002   # Random sent saturation point (92.4% ASR)
POISON_RATE_SENT_CBS    = 0.01    # CBS sent saturation point (93.0% ASR)
WORD_TRIGGER = "cf"
SENT_TRIGGER = "The absent gerbil filed a complaint downtown."
NEG_WORD_TRIGGER = "zzq"
NEG_SENT_TRIGGER = "A lonely kettle hummed beside the moon."
EVAL_SIZE = 10000   # subsample of the 38k test set; raise to full for final publication numbers
EPOCHS = 3
print(DEVICE)

cuda


In [6]:
ds = load_dataset("fancyzhx/yelp_polarity")
full_train_df = pd.DataFrame({"sentence": ds["train"]["text"], "label": ds["train"]["label"]})
clean_train_df = full_train_df.sample(n=TRAIN_SUBSAMPLE, random_state=SEED).reset_index(drop=True)

full_test_df = pd.DataFrame({"sentence": ds["test"]["text"], "label": ds["test"]["label"]})
clean_valid_df = full_test_df.sample(n=EVAL_SIZE, random_state=SEED).reset_index(drop=True)
print("train:", clean_train_df.shape, "| eval subsample:", clean_valid_df.shape)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_hf_dataset(df, tok=None):
    tok = tok or tokenizer
    d = Dataset.from_pandas(df[["sentence", "label"]].reset_index(drop=True))
    d = d.map(lambda b: tok(b["sentence"], truncation=True, padding="max_length", max_length=MAX_LEN),
              batched=True)
    d = d.rename_column("label", "labels")
    d.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
    return d

train: (25000, 2) | eval subsample: (10000, 2)


## Train

In [7]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(DEVICE)
train_ds = to_hf_dataset(clean_train_df)
valid_ds = to_hf_dataset(clean_valid_df)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    return {"accuracy": acc, "precision": p, "recall": r, "f1": f1}

args = TrainingArguments(
    output_dir="./results_e1_clean_yelp", num_train_epochs=EPOCHS,
    per_device_train_batch_size=8, per_device_eval_batch_size=32,
    learning_rate=2e-5, eval_strategy="epoch", save_strategy="no",
    logging_steps=200, seed=SEED, report_to="none",
)
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=valid_ds,
                   compute_metrics=compute_metrics)
trainer.train()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.158988,0.159105,0.951900,0.955254,0.947770,0.951497
2,0.094204,0.232070,0.952600,0.945411,0.960225,0.952761
3,0.026205,0.275387,0.954400,0.955479,0.952792,0.954134


TrainOutput(global_step=9375, training_loss=0.12033621559143066, metrics={'train_runtime': 2572.1433, 'train_samples_per_second': 29.159, 'train_steps_per_second': 3.645, 'total_flos': 9866664576000000.0, 'train_loss': 0.12033621559143066, 'epoch': 3.0})

## Evaluate + save
Reused as: (a) E1 baseline, (b) surrogate for E3's CBS scoring.

In [8]:
preds = np.argmax(trainer.predict(valid_ds).predictions, axis=-1)
cacc = accuracy_score(clean_valid_df["label"], preds)
p, r, f1, _ = precision_recall_fscore_support(clean_valid_df["label"], preds, average="binary")
e1_results = {"CACC": cacc, "Precision": p, "Recall": r, "F1": f1}
print(e1_results)

model.save_pretrained("./models/e1_clean_yelp")
tokenizer.save_pretrained("./models/e1_clean_yelp")
print("saved e1_clean_yelp")

{'CACC': 0.9544, 'Precision': 0.9554794520547946, 'Recall': 0.9527922860586581, 'F1': 0.9541339770669885}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved e1_clean_yelp


In [ ]:
results_df = pd.DataFrame([e1_results])
results_df.to_json("./results_e1_clean_yelp.json", index=False)

: 